In [7]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- Path Configuration ---
input_dir = r"C:\Users\28616\Desktop\Filtered_Matrices"
sample_ratio_dir = r"C:\Users\28616\Desktop\Sample_Ratio_Matrices"
os.makedirs(sample_ratio_dir, exist_ok=True)

sample_ids = ["49", "50", "51", "52", "53", "54"]

# 1. 预扫描：获取所有 Cluster 出现过的位点并集 (Union)
print("Step 1: Scanning all unique sites (Union)...")
all_site_union = set()
for n in range(6):
    file_path = os.path.join(input_dir, f"Cluster_{n}_new.csv")
    if os.path.exists(file_path):
        cols = pd.read_csv(file_path, index_col=0, nrows=0).columns
        all_site_union.update(cols)

union_list = sorted(list(all_site_union))
print(f"Total unique sites in union: {len(union_list)}")

# 2. 按照并集重构，缺失部分自动补 0
for s_id in tqdm(sample_ids, desc="Step 2: Reconstructing Sample Matrices"):
    sample_data_dict = {}
    
    for n in range(6):
        file_path = os.path.join(input_dir, f"Cluster_{n}_new.csv")
        df = pd.read_csv(file_path, index_col=0)
        
        # 计算该 Cluster 现有的 Ratio
        g = df.loc[f"{s_id}_G"]
        a = df.loc[f"{s_id}_A"]
        ratio_series = g / (g + a + 1)
        
        # 使用 reindex 强制对齐到并集名单，缺失的位点自动填 0
        aligned_ratio = ratio_series.reindex(union_list, fill_value=0)
        sample_data_dict[f"Cluster_{n}"] = aligned_ratio
    
    # 转换为 DataFrame: 行是 Cluster, 列是位点
    df_sample = pd.DataFrame(sample_data_dict).T
    
    # 保存
    save_path = os.path.join(sample_ratio_dir, f"Sample_{s_id}_ratio_matrix.csv")
    df_sample.to_csv(save_path)

print("\nSuccess: All sample matrices are reconstructed and aligned to union sites.")
print(f"Matrix shape for each sample: (6 Clusters, {len(union_list)} Sites)")

Step 1: Scanning all unique sites (Union)...
Total unique sites in union: 4419


Step 2: Reconstructing Sample Matrices: 100%|████████████████████████████████████████████| 6/6 [00:01<00:00,  4.29it/s]


Success: All sample matrices are reconstructed and aligned to union sites.
Matrix shape for each sample: (6 Clusters, 4419 Sites)
